In [1]:
import numpy as np
from LanzaModels import TIK_1D
from ADMMsRustici import SpectralSolver
from signalClass import *
import time

In [2]:
np.random.seed(24102001)
n = 1024

construct blur matrix

In [3]:
#blur matrix construction

a = 0.25
b = 0.5
c = 0.25

diagB = b * np.ones(shape=(n,))
offDiagA = a * np.ones(shape=(n-1,))
offDiagC = c * np.ones(shape=(n-1,))
A = np.diag(diagB, 0) + np.diag(offDiagC, 1) + np.diag(offDiagA, -1)

#apply anti-reflexive BCs

A[0][0] = 2 * a + b
A[0][1] = c - a
A[n-1][n-2] = a - c
A[n-1][n-1] = b + 2 * c

#end blur matrix construction

construct signal

In [4]:
#begin signal construction

PwSignal = signal(n)
RndSignal = signal(n)
sigma = 0.01

PwSignal.generate_cartoon_sign(2, 100)
RndSignal.generate_GG_realization(0, sigma, 2)

xTrue = PwSignal.get_image()
xCorrupted = (A @ xTrue) + RndSignal.get_image()

#end signal construction

Define the TVL2 model

In [5]:
mu = 2
VarModel = TIK_1D.TIK_1DClass(A, xCorrupted, mu)

Now, we need to initialize and define the solver

In [6]:
#begin solver construction

xk = np.copy(xCorrupted)
yk = np.random.randn(n,)
betak = 1
lk = np.zeros(n)

MySolver = SpectralSolver.SpectralSolverClass(VarModel, xk, yk, lk, betak)

#end solver construction

In [7]:
iters = 20

XsolutionHistory = np.zeros(shape=(iters, n))
YsolutionHistory = np.zeros(shape=(iters, n))

lambdaHistory = np.zeros(shape=(iters, n))

betaHistory = np.zeros(shape=(iters,))

PrimalResidueHistory = np.zeros(shape=(iters,))
DualResidueHistory = np.zeros(shape=(iters,))

CpuTimes = np.zeros(shape=(iters,))

In [8]:
for iter in range(0, iters):

    print(f"{iter} / {iters}")

    sTime = time.process_time_ns()

    xk_1, yk_1, lk_1, betak_1 = MySolver.CallIterationStep(xk, yk, lk, betak)

    eTime = time.process_time_ns()

    
    primalResidue = np.linalg.norm(VarModel.P @ xk_1 + VarModel.Q @ yk_1 - VarModel.c)
    dualResidue = np.abs(
                (VarModel.mu / 2) * (VarModel.fidelity(xk_1) - VarModel.fidelity(xk) ) + \
                VarModel.regularizer(yk_1) - VarModel.regularizer(yk)							
                        )

    XsolutionHistory[iter, :] = xk
    YsolutionHistory[iter, :] = yk
    lambdaHistory[iter, :] = lk
    betaHistory[iter] = betak

    PrimalResidueHistory[iter] = primalResidue
    DualResidueHistory[iter] = dualResidue
    CpuTimes[iter] = ( (eTime - sTime) / 1e9 ) + CpuTimes[iter - 1]

    xk = xk_1
    yk = yk_1
    lk = lk_1
    betak = betak_1


0 / 20
1 / 20
2 / 20
3 / 20
4 / 20
5 / 20
6 / 20
7 / 20
8 / 20
9 / 20
10 / 20
11 / 20
12 / 20
13 / 20
14 / 20
15 / 20
16 / 20
17 / 20
18 / 20
19 / 20


In [9]:

xReconstr = XsolutionHistory[iters - 1, :]
ConvergenceDistance = np.zeros(shape=(iters,))

ConvergenceDistance = np.linalg.norm(XsolutionHistory - xReconstr, axis=1)

In [10]:
np.savez_compressed(
    "./SpectralADMMTIK-Gauss.npz",
    Xs = XsolutionHistory,
    Ys = YsolutionHistory,
    Ls = lambdaHistory,
    Betas = betaHistory,

    PrimalRes = PrimalResidueHistory,
    DualRes = DualResidueHistory,

    ConvergenceDistance = ConvergenceDistance,
    CpuTimes = CpuTimes,

    xTrue = xTrue,
    xCorrupted = xCorrupted,
)